In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, split, explode, size, array_contains,
    create_map, map_keys, map_values,
    udf, upper, regexp_replace, concat, lit, trim
)

spark = SparkSession.builder.appName("Week2_Day5").master("local[*]").getOrCreate()

# Products with comma-separated tags (very common in real data)
product_data = [
    (1, "Gaming Laptop",   1200.0, "electronics,gaming,computers"),
    (2, "Office Chair",     350.0, "furniture,office"),
    (3, "Mechanical KB",    150.0, "electronics,gaming,accessories"),
    (4, "Standing Desk",    600.0, "furniture,office,ergonomic"),
    (5, "Webcam",            80.0, "electronics,office,accessories"),
    (6, "Monitor",          400.0, "electronics,computers"),
]

product_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("tags", StringType(), False),   # Comma-separated string!
])

df_products = spark.createDataFrame(product_data, product_schema)

# Customers with contact info as a map
customer_data = [
    (1, "Alice",   {"phone": "0771234567", "city": "Kampala",  "country": "Uganda"}),
    (2, "Bob",     {"phone": "0789876543", "city": "Nairobi",  "country": "Kenya"}),
    (3, "Charlie", {"phone": "(077) 555-1234", "city": "Kigali", "country": "Rwanda"}),
    (4, "Diana",   {"phone": "077.999.8888", "city": "Kampala", "country": "Uganda"}),
]

customer_schema = StructType([
    StructField("cust_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("contact", MapType(StringType(), StringType()), False),
])

df_customers = spark.createDataFrame(customer_data, customer_schema)

In [3]:
df_with_array = df_products.withColumn(
    "tags_array",
    split(col("tags"), ",")   # Split on comma
)

df_with_array.select("name", "tags", "tags_array").show(truncate=False)

+-------------+------------------------------+----------------------------------+
|name         |tags                          |tags_array                        |
+-------------+------------------------------+----------------------------------+
|Gaming Laptop|electronics,gaming,computers  |[electronics, gaming, computers]  |
|Office Chair |furniture,office              |[furniture, office]               |
|Mechanical KB|electronics,gaming,accessories|[electronics, gaming, accessories]|
|Standing Desk|furniture,office,ergonomic    |[furniture, office, ergonomic]    |
|Webcam       |electronics,office,accessories|[electronics, office, accessories]|
|Monitor      |electronics,computers         |[electronics, computers]          |
+-------------+------------------------------+----------------------------------+



In [4]:
df_array_ops = df_with_array.select(
    "name",
    "tags_array",
    size("tags_array").alias("num_tags"),                          # How many tags?
    array_contains("tags_array", "gaming").alias("is_gaming"),     # Contains "gaming"?
    col("tags_array").getItem(0).alias("first_tag"),               # First element
    col("tags_array").getItem(1).alias("second_tag"),              # Second element
)

df_array_ops.show(truncate=False)

+-------------+----------------------------------+--------+---------+-----------+----------+
|name         |tags_array                        |num_tags|is_gaming|first_tag  |second_tag|
+-------------+----------------------------------+--------+---------+-----------+----------+
|Gaming Laptop|[electronics, gaming, computers]  |3       |true     |electronics|gaming    |
|Office Chair |[furniture, office]               |2       |false    |furniture  |office    |
|Mechanical KB|[electronics, gaming, accessories]|3       |true     |electronics|gaming    |
|Standing Desk|[furniture, office, ergonomic]    |3       |false    |furniture  |office    |
|Webcam       |[electronics, office, accessories]|3       |false    |electronics|office    |
|Monitor      |[electronics, computers]          |2       |false    |electronics|computers |
+-------------+----------------------------------+--------+---------+-----------+----------+



In [5]:
df_with_array.filter(array_contains(col("tags_array"), "gaming")).select("name", "price").show()

+-------------+------+
|         name| price|
+-------------+------+
|Gaming Laptop|1200.0|
|Mechanical KB| 150.0|
+-------------+------+



In [6]:
df_exploded = df_with_array.select(
    "product_id",
    "name",
    "price",
    explode("tags_array").alias("tag")   # Each tag gets its own row
)

df_exploded.show(truncate=False)

+----------+-------------+------+-----------+
|product_id|name         |price |tag        |
+----------+-------------+------+-----------+
|1         |Gaming Laptop|1200.0|electronics|
|1         |Gaming Laptop|1200.0|gaming     |
|1         |Gaming Laptop|1200.0|computers  |
|2         |Office Chair |350.0 |furniture  |
|2         |Office Chair |350.0 |office     |
|3         |Mechanical KB|150.0 |electronics|
|3         |Mechanical KB|150.0 |gaming     |
|3         |Mechanical KB|150.0 |accessories|
|4         |Standing Desk|600.0 |furniture  |
|4         |Standing Desk|600.0 |office     |
|4         |Standing Desk|600.0 |ergonomic  |
|5         |Webcam       |80.0  |electronics|
|5         |Webcam       |80.0  |office     |
|5         |Webcam       |80.0  |accessories|
|6         |Monitor      |400.0 |electronics|
|6         |Monitor      |400.0 |computers  |
+----------+-------------+------+-----------+



In [8]:
from pyspark.sql.functions import count, avg, round
# Which tag has the most products? What's the average price per tag?
df_exploded.groupBy("tag").agg(
    count("product_id").alias("product_count"),
    round(avg("price"), 2).alias("avg_price")
).orderBy(col("product_count").desc()).show()

+-----------+-------------+---------+
|        tag|product_count|avg_price|
+-----------+-------------+---------+
|electronics|            4|    457.5|
|     office|            3|   343.33|
|  computers|            2|    800.0|
|     gaming|            2|    675.0|
|  furniture|            2|    475.0|
|accessories|            2|    115.0|
|  ergonomic|            1|    600.0|
+-----------+-------------+---------+



In [9]:
df_customers.show(truncate=False)

+-------+-------+------------------------------------------------------------+
|cust_id|name   |contact                                                     |
+-------+-------+------------------------------------------------------------+
|1      |Alice  |{country -> Uganda, city -> Kampala, phone -> 0771234567}   |
|2      |Bob    |{country -> Kenya, city -> Nairobi, phone -> 0789876543}    |
|3      |Charlie|{country -> Rwanda, city -> Kigali, phone -> (077) 555-1234}|
|4      |Diana  |{country -> Uganda, city -> Kampala, phone -> 077.999.8888} |
+-------+-------+------------------------------------------------------------+



In [10]:
df_contact_flat = df_customers.select(
    "cust_id",
    "name",
    col("contact").getItem("phone").alias("phone"),
    col("contact").getItem("city").alias("city"),
    col("contact").getItem("country").alias("country"),
)

df_contact_flat.show(truncate=False)

+-------+-------+--------------+-------+-------+
|cust_id|name   |phone         |city   |country|
+-------+-------+--------------+-------+-------+
|1      |Alice  |0771234567    |Kampala|Uganda |
|2      |Bob    |0789876543    |Nairobi|Kenya  |
|3      |Charlie|(077) 555-1234|Kigali |Rwanda |
|4      |Diana  |077.999.8888  |Kampala|Uganda |
+-------+-------+--------------+-------+-------+



In [11]:
df_customers.select(
    "name",
    map_keys("contact").alias("all_keys"),
    map_values("contact").alias("all_values")
).show(truncate=False)

+-------+----------------------+--------------------------------+
|name   |all_keys              |all_values                      |
+-------+----------------------+--------------------------------+
|Alice  |[country, city, phone]|[Uganda, Kampala, 0771234567]   |
|Bob    |[country, city, phone]|[Kenya, Nairobi, 0789876543]    |
|Charlie|[country, city, phone]|[Rwanda, Kigali, (077) 555-1234]|
|Diana  |[country, city, phone]|[Uganda, Kampala, 077.999.8888] |
+-------+----------------------+--------------------------------+



In [12]:
from pyspark.sql.functions import explode

df_customers.select(
    "name",
    explode("contact")   # Creates "key" and "value" columns automatically
).show(truncate=False)

+-------+-------+--------------+
|name   |key    |value         |
+-------+-------+--------------+
|Alice  |country|Uganda        |
|Alice  |city   |Kampala       |
|Alice  |phone  |0771234567    |
|Bob    |country|Kenya         |
|Bob    |city   |Nairobi       |
|Bob    |phone  |0789876543    |
|Charlie|country|Rwanda        |
|Charlie|city   |Kigali        |
|Charlie|phone  |(077) 555-1234|
|Diana  |country|Uganda        |
|Diana  |city   |Kampala       |
|Diana  |phone  |077.999.8888  |
+-------+-------+--------------+



In [13]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Step 1: Write a normal Python function
def clean_phone(phone_str):
    """Remove everything except digits"""
    if phone_str is None:
        return None
    return ''.join(char for char in phone_str if char.isdigit())

# Step 2: Register it as a Spark UDF
clean_phone_udf = udf(clean_phone, StringType())

# Step 3: Use it like any other Spark function
df_clean_phones = df_contact_flat.withColumn(
    "phone_clean",
    clean_phone_udf(col("phone"))
)

df_clean_phones.select("name", "phone", "phone_clean").show(truncate=False)

+-------+--------------+-----------+
|name   |phone         |phone_clean|
+-------+--------------+-----------+
|Alice  |0771234567    |0771234567 |
|Bob    |0789876543    |0789876543 |
|Charlie|(077) 555-1234|0775551234 |
|Diana  |077.999.8888  |0779998888 |
+-------+--------------+-----------+

